# 1. Getting Started

## Initialize Foundry Manager and Execution Providers

Foundry Local runs on top of ONNX runtime.<br>
Before we start, we should download and register [ONNX execution providers](https://onnxruntime.ai/docs/execution-providers/), depending on your hardware acceleration. (In my case, I have used a local machine with NVIDIA GPU and its driver.)

For the first time, it will take time, because of the initial download.

In [1]:
from foundry_local_sdk import Configuration, FoundryLocalManager

config = Configuration(app_name="foundry_local_samples")
FoundryLocalManager.initialize(config)
manager = FoundryLocalManager.instance

current_ep = ""
def ep_progress(ep_name: str, percent: float):
    global current_ep
    if ep_name != current_ep:
        if current_ep:
            print()
        current_ep = ep_name
    print(f"\r  {ep_name:<30}  {percent:5.1f}%", end="", flush=True)

result = manager.download_and_register_eps(progress_callback=ep_progress)

  CUDAExecutionProvider           100.0%
  NvTensorRTRTXExecutionProvider  100.0%

In [2]:
result

EpDownloadResult(success=True, status='EP registration complete: 2 bootstrapper(s) succeeded in 785ms. Available EPs: CPUExecutionProvider, WebGpuExecutionProvider, CUDAExecutionProvider', registered_eps=['CUDAExecutionProvider', 'NvTensorRTRTXExecutionProvider'], failed_eps=[])

## List available models

Now let's list available models.

In my case, a lot of models in the list are optimized for CUDA, because my environment is accelerated by NVIDIA GPU.<br>
For example, ```qwen2.5-0.5b-instruct-cuda-gpu``` is obtained as a model with alias "qwen2.5-0.5b".

> Note : The model's list is refreshed every 6 hours.

In [3]:
models = manager.catalog.list_models()
for m in models:
    print(f"{m.id} (alias: {m.alias})")

qwen2.5-coder-0.5b-instruct-cuda-gpu:4 (alias: qwen2.5-coder-0.5b)
Phi-4-mini-reasoning-cuda-gpu:3 (alias: phi-4-mini-reasoning)
qwen2.5-0.5b-instruct-cuda-gpu:4 (alias: qwen2.5-0.5b)
qwen2.5-1.5b-instruct-cuda-gpu:4 (alias: qwen2.5-1.5b)
qwen2.5-coder-1.5b-instruct-cuda-gpu:4 (alias: qwen2.5-coder-1.5b)
Phi-4-mini-instruct-cuda-gpu:5 (alias: phi-4-mini)
qwen2.5-14b-instruct-cuda-gpu:4 (alias: qwen2.5-14b)
qwen2.5-coder-14b-instruct-cuda-gpu:4 (alias: qwen2.5-coder-14b)
qwen2.5-coder-7b-instruct-cuda-gpu:4 (alias: qwen2.5-coder-7b)
qwen2.5-7b-instruct-cuda-gpu:4 (alias: qwen2.5-7b)
openai-whisper-large-v3-turbo-cuda-gpu:2 (alias: whisper-large-v3-turbo)
gpt-oss-20b-cuda-gpu:1 (alias: gpt-oss-20b)
Phi-3-mini-128k-instruct-cuda-gpu:2 (alias: phi-3-mini-128k)
Phi-3.5-mini-instruct-cuda-gpu:2 (alias: phi-3.5-mini)
Phi-4-cuda-gpu:2 (alias: phi-4)
deepseek-r1-distill-qwen-7b-cuda-gpu:4 (alias: deepseek-r1-7b)
openai-whisper-small-cuda-gpu:2 (alias: whisper-small)
Phi-3-mini-4k-instruct-cuda-

## Multi-turn chat

Now let's run chat completion with the accelerated model of "qwen2.5-0.5b".

The following ```client``` object is a client with the same specifications as OpenAI client, but it does not invoke HTTP REST endpoint, and instead directly calls the Foundry core and ONNX runtime library.

For the first time, it will take time, because of the initial download.

In [4]:
import json

model = manager.catalog.get_model("qwen2.5-0.5b")
model.download()
model.load()

client = model.get_chat_client()
completion = client.complete_chat([
    {
        "role": "system",
        "content": "You are a helpful assistant, named Foundry AI."
    },
    {
        "role": "user",
        "content": "Hello, Foundry AI"
    }
])

print(json.dumps(completion.choices[0].delta, ensure_ascii=False, indent=4, sort_keys=True, separators=(',', ': ')))

{
    "content": "Hello! How may I assist you with foundry AI? What can I help you do today?",
    "role": "assistant",
    "tool_calls": []
}


## Streaming Response

Same as OpenAI API, you can get response as streaming.<br>
In production, you can invoke the streaming for the fluent experience.

In [5]:
for chunk in client.complete_streaming_chat([
    {
        "role": "system",
        "content": "You are a helpful assistant, named Foundry AI."
    },
    {
        "role": "user",
        "content": "Hello, Foundry AI"
    }
]):
    content = chunk.choices[0].delta.content
    print(content, end="", flush=True)

print()

Hello!! I'm here to help you whenever you need it. What can I assist you with?


## Local function calling

You can also work with tool calling.<br>
Here, let's work with a simple local function calling. (For remote MCP tool calling, see [Lesson 3](./03_agent_framework.ipynb).)

> Note : Avoid complex reasoning in small language models. (Use simple tool calling.)

First, let's define functions as local tools.

In [ ]:
from random import randint

def get_weather(location) -> str:
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return conditions[randint(0, 3)]

def get_temperature(location) -> str:
    return f"{randint(10, 30)} degrees"

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the weather for a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "the location to get the weather for"
                    },
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_temperature",
            "description": "Get the temperature for a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "the location to get the temperature for"
                    }
                },
                "required": ["location"]
            }
        }
    }
]

Let's invoke chat completion with above tool's definition.

In [8]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant with access to tools. "
                   "Use them when needed to answer questions accurately."
    },
    {
        "role": "user",
        "content": "Tell me the weather in Seattle."
    }
]

completion = client.complete_chat(
    messages = messages,
    tools = tools,
)

Now let's see the response.

In this example, I'm asking for the weather in Seattle.<br>
Therefore, above ```get_weather``` function will be used as a tool calling.

In [9]:
completion.choices[0].finish_reason

'tool_calls'

In [10]:
assistant_msg = {
    "role": "assistant",
    "content": completion.choices[0].message.content,
    "tool_calls": [
        {
            "id": tc.id,
            "type": tc.type,
            "function": {
                "name": tc.function.name,
                "arguments": tc.function.arguments,
            },
        }
        for tc in completion.choices[0].message.tool_calls
    ],
}
messages.append(assistant_msg)

print(json.dumps(messages, ensure_ascii=False, indent=4, sort_keys=True, separators=(',', ': ')))

[
    {
        "content": "You are a helpful assistant with access to tools. Use them when needed to answer questions accurately.",
        "role": "system"
    },
    {
        "content": "Tell me the weather in Seattle.",
        "role": "user"
    },
    {
        "content": "<tool_call>\n{\"name\": \"get_weather\", \"arguments\": {\"location\": \"Seattle\"}}\n</tool_call>",
        "role": "assistant",
        "tool_calls": [
            {
                "function": {
                    "arguments": "{\r\n  \"location\": \"Seattle\"\r\n}",
                    "name": "get_weather"
                },
                "id": "pBFXnaM6d",
                "type": "function"
            }
        ]
    }
]


Now let's invoke tools (in this example, ```get_weather``` function), and add result in chat messages (chat history).

In [11]:
tool_functions = {
    "get_weather": get_weather,
    "get_temperature": get_temperature
}

choice = completion.choices[0].message
for tool_call in choice.tool_calls:
    function_name = tool_call.function.name
    func = tool_functions[function_name]
    args = json.loads(tool_call.function.arguments)
    result = func(**args)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result
    })

print(json.dumps(messages, ensure_ascii=False, indent=4, sort_keys=True, separators=(',', ': ')))

[
    {
        "content": "You are a helpful assistant with access to tools. Use them when needed to answer questions accurately.",
        "role": "system"
    },
    {
        "content": "Tell me the weather in Seattle.",
        "role": "user"
    },
    {
        "content": "<tool_call>\n{\"name\": \"get_weather\", \"arguments\": {\"location\": \"Seattle\"}}\n</tool_call>",
        "role": "assistant",
        "tool_calls": [
            {
                "function": {
                    "arguments": "{\r\n  \"location\": \"Seattle\"\r\n}",
                    "name": "get_weather"
                },
                "id": "pBFXnaM6d",
                "type": "function"
            }
        ]
    },
    {
        "content": "sunny",
        "role": "tool",
        "tool_call_id": "pBFXnaM6d"
    }
]


Invoke chat completion again.

In [12]:
completion = client.complete_chat(
    messages = messages,
    tools = tools,
)

The language model then returns the final response.

In [13]:
completion.choices[0].finish_reason

'stop'

In [14]:
completion.choices[0].message.content

'The weather in Seattle is sunny.'